# PDF Data Extraction with Open-Weight Models

**2026 UIUC Workshop -- de Medeiros Insect Flower Visitors Lab, Field Museum**

This is the 2026 rewrite of the 2025 ESA workshop notebook
(`pdf_data_extraction.ipynb`, Claude Haiku 4.5 via AWS Bedrock). Everything
here runs on **open-weight models**, **free on Google Colab**, with **no API
key to distribute**.

Audience: systematic entomologists, not software engineers. No programming
background beyond basic Python is assumed -- if a cell looks unfamiliar,
treat that as a cue to ask, not a sign that you are behind.

Repository: `de-Medeiros-insect-lab/2026_UIUC_workshop_llm_pdf_extraction`


## 0. Why this changed since 2025

Three things are different this year, and each one removes or replaces a
whole section of the old notebook.

1. **No key distribution.** 2025 ran Claude through AWS Bedrock on the
   instructor's credits, and every student needed an individually issued
   key. Open models on Ollama need no key at all -- that deletes the entire
   setup ceremony, and the anxiety about accidentally spending someone
   else's grant money.
2. **Structured output is enforced, not negotiated.** 2025 spent several
   cells coaxing Claude into valid JSON, then repairing whatever came back.
   Ollama's `format=` parameter constrains decoding to a JSON Schema, so
   non-conforming output is not merely discouraged -- it is
   unrepresentable. Later today, Section 6 replaces prompt-wrangling with
   schema *design*.
3. **PDFs stop being free.** Claude read PDFs natively; the models we use
   today do not. We have to confront directly that a born-digital paper has
   a text layer, while a scan is only pixels until something reads it.
   That distinction turns out to be the most useful thing in this
   notebook, and Sections 3 and 4 are built entirely around it.

There is also a case for running models locally at all, beyond dodging API
keys: you can archive open weights alongside your data when you publish a
methods section, which you cannot do with a commercial API endpoint that may
not exist in five years. Unpublished specimen records and localities for
endangered taxa also never have to leave your machine.


## 1. Ollama on Colab

We will run everything through [Ollama](https://ollama.com), a small server
that manages open-weight models on your own machine (or, today, your Colab
VM) and exposes them through a simple API. Two models, one job each:

| model | role |
| --- | --- |
| `qwen3.5:9b` | reasoning, extraction, tool use -- the model you will talk to all day |
| `deepseek-ocr` | a dedicated transcription model for reading scanned pages |

The setup cell below installs Ollama and starts its server *only* when
running on Colab; locally it is a no-op, because you already have both. Run
it either way and keep reading -- on a fresh Colab VM the first run downloads
real weights and takes a couple of minutes.


In [ ]:
# Colab setup. On a fresh VM this takes ~2 minutes; keep reading while it runs.
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pydantic pandas
    !git clone -q https://github.com/de-Medeiros-insect-lab/2026_UIUC_workshop_llm_pdf_extraction.git
    os.chdir("2026_UIUC_workshop_llm_pdf_extraction")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from workshop_lib import server_ready, CHAT_MODEL, OCR_MODEL
assert server_ready(120), "Ollama did not start"
print("Ollama is up")


### Check for a GPU

These models are small by LLM standards, but still far too slow on a CPU for
a live workshop. Run this now, before we go any further.


In [ ]:
import subprocess

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True)
    have_gpu = gpu.returncode == 0
except FileNotFoundError:
    # No nvidia-smi binary at all -- e.g. a Mac, or a Colab CPU-only runtime.
    have_gpu = False

if have_gpu:
    print("GPU:", gpu.stdout.strip())
else:
    print("NO GPU on this runtime.")
    print("Colab often refuses free GPUs at busy times of day.")
    print("Please pair up with a neighbour who has one -- the models are far")
    print("too slow on CPU for a workshop.")


### Pull the models

About 14 GB combined. Start this now -- we will keep talking while it
downloads in the background.


In [ ]:
# ~14 GB total. Start this now; we will talk while it downloads.
!ollama pull {CHAT_MODEL}
!ollama pull {OCR_MODEL}
!ollama list


## 2. Messages, system prompts, and role-play

Chat models are driven by a list of *messages*, each tagged with a role:
`system` (instructions that set the model's persona and constraints),
`user` (what you ask), and `assistant` (what the model answers). The
`system` message is not decoration -- it is the single strongest lever you
have over the model's behaviour, and it works because a chat model is
fundamentally playing a role you assign it:

> Shanahan, M., McDonell, K. & Reynolds, L. *Role play with large language
> models.* Nature 623, 493-498 (2023).

Their argument, in short: a chat model does not "become" an expert
coleopterist because you told it so -- it predicts what an expert
coleopterist's reply would look like, and a good system prompt is good stage
direction. The effect below is not a party trick; it is the mechanism.


In [ ]:
import ollama

reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system",
         "content": "You are an expert coleopterist. Answer in two sentences."},
        {"role": "user",
         "content": "What is a rostrum, and which beetles have one?"},
    ],
    think=False,
    options={"temperature": 0},
)
print(reply.message.content)


### Hands-on 1

Write your own system prompt. Pick a persona and a question -- a curator
sorting a drawer of unidentified weevils, a strict referee checking a species
description, a museum docent explaining a specimen to visitors, whatever you
like -- and see how far the answer moves.


In [ ]:
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "WRITE A ROLE FOR THE MODEL HERE"},
        {"role": "user",   "content": "ASK YOUR QUESTION HERE"},
    ],
    think=False,
    options={"temperature": 0},
)
print(reply.message.content)


**Now run the same question twice more:** once with `"content": ""` for the
system message (no persona at all), and once with a long, detailed persona.
Compare the three answers. Hosted models like the one used in 2025 already
answer reasonably with no system prompt; a 9B model like this one needs much
firmer steering, and the gap between "no persona" and "detailed persona" is
larger than you might expect.


## 3. PDFs are not text: text layer vs. pixels

A PDF page is not "text" -- it is a page of ink positions, and text only
comes along for the ride when the file happens to carry an embedded text
layer. Two very different documents sit in `example_pdfs/`:

- **`deMedeiros2013Zootaxa.pdf`** -- born-digital, 2013, three
  *Anchylorhynchus* weevil species. Its text layer is exact.
- **`Marshall1929_AnnMagNatHist.pdf`** -- an 8-page scan of a 1929 paper. It
  *also* has a text layer (someone ran OCR on it at some point), but that
  layer can be silently wrong.

`workshop_lib.open_pdf` and `get_page_text` (pages are 1-based, like a real
paginated document) get you the text layer for free -- when it is
trustworthy.


In [ ]:
from workshop_lib import open_pdf, get_page_text, render_page
from IPython.display import Image, display
import base64

modern = open_pdf("example_pdfs/deMedeiros2013Zootaxa.pdf")
legacy = open_pdf("example_pdfs/Marshall1929_AnnMagNatHist.pdf")

print("MODERN, born-digital:")
print(repr(get_page_text(modern, 1)[:200]))
print("\nLEGACY, a 1929 scan -- note this is NOT empty:")
print(repr(get_page_text(legacy, 5)[:200]))


The legacy page's text is not empty, and it reads as plausible prose -- that
is exactly what makes silently-corrupt OCR the dangerous case rather than the
obvious one. Look at the page image itself and judge for yourself:


In [ ]:
display(Image(data=base64.b64decode(render_page(legacy, 5, dpi=100))))


The text layer said `Cureulionidse`; the page plainly reads `Curculionidae`.
Nothing raised an exception and no field came back empty -- an extraction
pipeline trusting the text layer alone would confidently attribute this
species to a family that matches no taxonomic authority anywhere. Later
today, Section 4 teaches the model to read pages like this one from their
pixels instead of trusting the text layer.


## 5. Thinking

`qwen3.5:9b` is a *reasoning* model: pass `think=True` and it produces a
chain of thought (`reply.message.thinking`), reported separately from its
final answer (`reply.message.content`). Here it works through a short
arithmetic problem out loud before answering:


In [ ]:
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user",
               "content": "A beetle is 4.2 mm long and 1.4 mm wide. "
                          "What is the length-to-width ratio?"}],
    think=True,
    options={"temperature": 0},
)
print("--- reasoning ---");  print(reply.message.thinking)
print("--- answer ---");     print(reply.message.content)


**This setting is not free, and it is not always a win.** Passing
`think=True` on a *transcription* task -- read this page and copy the text --
made this same model emit 123,055 characters of reasoning, run into its own
output-length limit, and return no answer at all. Transcription is recall:
there is nothing on the page to reason *about*, so reasoning has nothing to
do except ruminate. That is why every transcription and one-shot extraction
call in this notebook passes `think=False`.

Hold onto that thought -- it is not the end of the story. Section 7 puts this
same model in charge of *deciding* whether a page's text can be trusted, and
there the setting flips: judging trustworthiness is reasoning work, and the
loop does not function without `think=True`. The transferable skill is not
"always start with this setting off" -- it is recognising which kind of task
is in front of you.
